<a href="https://www.kaggle.com/code/asgx1224/hftbacktest-first?scriptVersionId=356597778" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import sys

print("Python:", sys.version)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
%pip install -q --disable-pip-version-check hftbacktest==2.4.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 31.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.68.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import hftbacktest
import numpy as np
import numba

print("hftbacktest:", hftbacktest.__version__)
print("numpy:", np.__version__)
print("numba:", numba.__version__)

hftbacktest: 2.4.4
numpy: 2.0.2
numba: 0.68.0


In [4]:
from pathlib import Path
import numpy as np

DATA_PATH = Path(
    "/kaggle/input/datasets/asgx1224/hftbacktest/BTCUSDT_2023-04-01.npz"
)

print("exists:", DATA_PATH.exists())
print("path:", DATA_PATH)

if not DATA_PATH.exists():
    print("\nParent exists:", DATA_PATH.parent.exists())
    if DATA_PATH.parent.exists():
        print("Files nearby:")
        for p in DATA_PATH.parent.iterdir():
            print(" ", p.name)

exists: True
path: /kaggle/input/datasets/asgx1224/hftbacktest/BTCUSDT_2023-04-01.npz


In [5]:
npz = np.load(DATA_PATH)

print("NPZ keys:", npz.files)

data = npz["data"]

print("shape:", data.shape)
print("dtype:")
print(data.dtype)

print("\nfirst 5 rows:")
print(data[:5])

NPZ keys: ['data']
shape: (60943533,)
dtype:
[('ev', '<u8'), ('exch_ts', '<i8'), ('local_ts', '<i8'), ('px', '<f8'), ('qty', '<f8'), ('order_id', '<u8'), ('ival', '<i8'), ('fval', '<f8')]

first 5 rows:
[(3489660930, 1680307200158000000, 1680307200165963000, 28454.8, 0.009, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200173448000, 28454.8, 0.027, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200173512000, 28454.8, 0.005, 0, 0, 0.)
 (3489660930, 1680307200165000000, 1680307200174315000, 28454.8, 0.022, 0, 0, 0.)
 (3489660930, 1680307200183000000, 1680307200190901000, 28454.8, 0.004, 0, 0, 0.)]


In [6]:
print("rows:", len(data))

print("\nTimestamp range:")
print("exch_ts:", data["exch_ts"].min(), "->", data["exch_ts"].max())
print("local_ts:", data["local_ts"].min(), "->", data["local_ts"].max())

print("\nUnique ev values:")
ev_values, ev_counts = np.unique(data["ev"], return_counts=True)

for ev, count in zip(ev_values, ev_counts):
    print(f"ev={ev:<12} count={count:,}")

rows: 60943533

Timestamp range:
exch_ts: 1680307200158000000 -> 1680393599990000000
local_ts: 1680307200165963000 -> 1680393599993357000

Unique ev values:
ev=1342177282   count=186,292
ev=1610612738   count=185,510
ev=2415919106   count=186,292
ev=2684354562   count=185,510
ev=3489660929   count=28,519,971
ev=3489660930   count=1,014,353
ev=3489660931   count=1
ev=3489660932   count=1,000
ev=3758096385   count=29,648,277
ev=3758096386   count=1,015,325
ev=3758096387   count=1
ev=3758096388   count=1,001


In [7]:
latency_ns = data["local_ts"] - data["exch_ts"]

print("Original Tardis local_ts - exch_ts:")
print("min :", latency_ns.min(), "ns")
print("p50 :", np.percentile(latency_ns, 50), "ns")
print("p95 :", np.percentile(latency_ns, 95), "ns")
print("p99 :", np.percentile(latency_ns, 99), "ns")
print("max :", latency_ns.max(), "ns")

Original Tardis local_ts - exch_ts:
min : 1298000 ns
p50 : 2491000.0 ns
p95 : 34378000.0 ns
p99 : 175008000.0 ns
max : 2465107000 ns
